# Day 4 — Feature engineering, imbalance handling and model selection

**Project:** RetailMind AI — Customer Churn Prediction

**Goal:** Create domain-based customer features, train the feature-engineered Random Forest and XGBoost models, and compare them with the tuned classical champion.


## 8. Day 4 — Domain feature engineering

Define a custom scikit-learn transformer that creates additional customer-level features from the existing columns. These engineered features represent business-relevant information in a form the models can use. Implementing the logic as a transformer allows it to be included in a pipeline and applied consistently to new data.

In [9]:
class RetailFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

        X["TenureGroup"] = pd.cut(
            X["tenure"], bins=[-1, 12, 24, 48, np.inf],
            labels=["New", "Medium", "Long", "Loyal"]
        ).astype("object")

        safe_tenure = X["tenure"].replace(0, np.nan)
        X["AverageMonthlySpend"] = X["TotalCharges"] / safe_tenure
        X["ChargeRatio"] = X["MonthlyCharges"] / (X["TotalCharges"] + 1)

        X["IsMonthToMonth"] = (X["Contract"] == "Month-to-month").astype(int)
        X["IsOneYearContract"] = (X["Contract"] == "One year").astype(int)
        X["IsTwoYearContract"] = (X["Contract"] == "Two year").astype(int)

        service_cols = [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        X["ServiceCount"] = sum((X[col] == "Yes").astype(int) for col in service_cols)
        X["HasInternet"] = (X["InternetService"] != "No").astype(int)
        X["HasPhone"] = (X["PhoneService"] == "Yes").astype(int)
        X["PaperlessBillingFlag"] = (X["PaperlessBilling"] == "Yes").astype(int)
        return X

# Inspect engineered columns. This is only a preview; pipelines below fit transforms on each fold.
feature_engineer = RetailFeatureEngineer()
X_train_engineered = feature_engineer.fit_transform(X_train)
display(X_train_engineered.head())
print("Feature count before:", X_train.shape[1])
print("Feature count after:", X_train_engineered.shape[1])


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TenureGroup,AverageMonthlySpend,ChargeRatio,IsMonthToMonth,IsOneYearContract,IsTwoYearContract,ServiceCount,HasInternet,HasPhone,PaperlessBillingFlag
3738,Male,0,No,No,35,No,No phone service,DSL,No,No,...,Long,48.618571,0.028896,1,0,0,3,1,0,0
3151,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,...,Medium,76.770000,0.065160,1,0,0,2,1,1,0
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,...,Medium,45.411538,0.068572,0,0,1,3,1,0,0
3867,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,...,Long,73.296154,0.038548,0,0,1,5,1,1,1
3810,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,...,New,44.550000,0.978046,1,0,0,1,1,1,0


Feature count before: 19
Feature count after: 29


## 9. Day 4 — Train Random Forest and XGBoost with feature selection

Train the Day 4 feature-engineered models, including Random Forest and XGBoost, with the feature-selection and class-imbalance handling specified in the project code. This step tests whether domain-specific features and an algorithm designed for boosted trees improve the ability to identify customers likely to churn.

In [10]:
# XGBoost is imported only for this Day 4 section.
from xgboost import XGBClassifier

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

def make_day4_pipeline(classifier):
    # Feature engineering and preprocessing are fitted within the pipeline,
    # avoiding leakage across cross-validation folds.
    sample_engineered = RetailFeatureEngineer().fit_transform(X_train)
    numeric_cols = sample_engineered.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = sample_engineered.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols)
    ])

    return Pipeline([
        ("feature_engineering", RetailFeatureEngineer()),
        ("preprocessor", prep),
        ("feature_selection", SelectKBest(score_func=f_classif, k=40)),
        ("classifier", classifier)
    ])

day4_models = {
    "Random Forest Day 4": RandomForestClassifier(
        n_estimators=250, max_depth=12, min_samples_split=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

day4_pipelines = {}
day4_results = []

for name, estimator in day4_models.items():
    print("Training:", name)
    pipe = make_day4_pipeline(estimator)

    # Determine a safe k based on transformed training data.
    X_eng = RetailFeatureEngineer().fit_transform(X_train)
    # Build a temporary preprocessor with the same columns to calculate encoded width.
    num_cols = X_eng.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X_eng.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    temp_preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])
    encoded_width = temp_preprocessor.fit_transform(X_eng).shape[1]
    pipe.set_params(feature_selection__k=min(40, encoded_width))

    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="recall", n_jobs=1)
    pipe.fit(X_train, y_train)
    day4_pipelines[name] = pipe

    metrics = evaluate_classifier(pipe, X_test, y_test, name)
    metrics["CV Recall"] = cv.mean()
    metrics["CV Recall Std"] = cv.std()
    day4_results.append(metrics)
    print(f"CV Recall: {cv.mean():.4f}")

day4_comparison = pd.DataFrame(day4_results).sort_values(
    ["CV Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(day4_comparison)
day4_comparison.to_csv(REPORTS_DIR / "day4_model_comparison.csv", index=False)


Training: Random Forest Day 4
CV Recall: 0.7324
Training: XGBoost
CV Recall: 0.7732


,Model,Accuracy,Precision,Recall,ROC-AUC,CV Recall,CV Recall Std
0,XGBoost,0.748758,0.518051,0.767380,0.841356,0.773244,0.011853
1,Random Forest Day 4,0.764372,0.541016,0.740642,0.837002,0.732441,0.008974


## 10. Day 4 — Compare the tuned Day 3 champion and select a classical champion

Evaluate the tuned Day 3 champion and the Day 4 candidates on the same untouched test set, then select the strongest classical model according to the project's comparison criteria. Using the same test rows makes the comparison fair; this selected model becomes the benchmark for the ANN on Day 5.

In [11]:
# Score the Day 3 model on the same untouched test set.
day3_metrics = evaluate_classifier(
    day3_champion, X_test, y_test, "Day 3 Tuned Champion"
)

classical_candidates = {
    "Day 3 Tuned Champion": day3_champion,
    **day4_pipelines
}

classical_comparison_rows = [day3_metrics]
for name, model in day4_pipelines.items():
    classical_comparison_rows.append(
        evaluate_classifier(model, X_test, y_test, name)
    )

classical_comparison = pd.DataFrame(classical_comparison_rows).sort_values(
    ["Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(classical_comparison)
classical_comparison.to_csv(REPORTS_DIR / "classical_champion_comparison.csv", index=False)

# Select by held-out recall for reporting/benchmarking. Treat this test-set choice
# as a final benchmark, not as a new hyperparameter-tuning loop.
classical_champion_name = classical_comparison.loc[0, "Model"]
classical_champion = classical_candidates[classical_champion_name]
joblib.dump(classical_champion, MODELS_DIR / "phase1_classical_champion.joblib")
print("Saved Phase 1 classical champion:", classical_champion_name)


,Model,Accuracy,Precision,Recall,ROC-AUC
0,Day 3 Tuned Champion,0.750177,0.519031,0.802139,0.840007
1,XGBoost,0.748758,0.518051,0.767380,0.841356
2,Random Forest Day 4,0.764372,0.541016,0.740642,0.837002


Saved Phase 1 classical champion: Day 3 Tuned Champion


## Day 4 — Completion

This notebook contains the executable project workflow through Day 4. Run all cells in order and review the printed metrics, tables and plots before recording results in the project README.